In [1]:
import requests
import tiktoken
import torch
from torch import nn
import datasets


tokenizer = tiktoken.get_encoding("gpt2")
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

cuda


In [2]:
engine_res = requests.get("https://github.com/Shanmukh-dev/Custom-GPT/raw/refs/heads/main/model.py")

with open("model.py", "w") as f:
    f.write(engine_res.text)
    # print(engine_res.text)
print("Downloaded model.py")

# Downloading train_utils.py
train_utils_res = requests.get("https://github.com/Shanmukh-dev/Custom-GPT/raw/refs/heads/main/train_utils.py")

with open("train_utils.py", "w") as f:
    f.write(train_utils_res.text)
    # print(engine_res.text)
print("Downloaded train_utils.py")

data_res = requests.get("https://github.com/Shanmukh-dev/Custom-GPT/raw/refs/heads/main/data.py")

with open("data.py", "w") as f:
    f.write(data_res.text)
    # print(engine_res.text)
print("Downloaded data.py")

#Downloading custom_gpt_v2.pth weights
wt_res = requests.get("https://github.com/Shanmukh-dev/Custom-GPT/raw/refs/heads/main/Models/tara_n1_pretrain_v2.pth")
with open("tara_n1_pretrain_v2.pth", "wb") as f:
    f.write(wt_res.content)
print("Downloaded modle weights")


Downloaded model.py
Downloaded train_utils.py
Downloaded data.py
Downloaded modle weights


In [3]:
from model import *
from train_utils import *
from data import *


config = GPTConfig(
    vocab_size=tokenizer.n_vocab,
    block_size=1024,
    batch_size=4,
    d_model=256,
    hidden_layers=1024,
    n_heads=4,
    n_layers=6,
)

# Mid Trainign with `SimpleStories/SimpleStories`, `pszemraj/simple_wikipedia_LM`, and  `rojagtap/bookcorpus` datasets

## Download and prep Dataset


In [4]:
from datasets import load_dataset

simple_stories_ds = load_dataset("SimpleStories/SimpleStories")
simple_wiki_ds = load_dataset("pszemraj/simple_wikipedia_LM")
bookcorpus_ds = load_dataset("rojagtap/bookcorpus")


README.md:   0%|          | 0.00/4.18k [00:00<?, ?B/s]

data/train-00000-of-00007.parquet: reconstructing file:   0%|          |  0.00B /  238MB            

data/train-00000-of-00007.parquet: downloading bytes:           |  0.00B            

data/train-00001-of-00007.parquet: reconstructing file:   0%|          |  0.00B /  238MB            

data/train-00001-of-00007.parquet: downloading bytes:           |  0.00B            

...

books_large_p1.txt: reconstructing file:   0%|          |  0.00B / 2.52GB            

books_large_p1.txt: downloading bytes:           |  0.00B            

books_large_p2.txt: reconstructing file:   0%|          |  0.00B / 2.10GB            

books_large_p2.txt: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/74004228 [00:00<?, ? examples/s]

In [5]:
print(len(simple_stories_ds))
print(len(simple_wiki_ds))
print(len(bookcorpus_ds))

2
3
1


In [7]:
bookcorpus_ds["train"][0:10]

{'text': ['the half-ling book one in the fall of igneeria series kaylee soderburg copyright 2013 kaylee soderburg all rights reserved .',
  'isbn : 1492913731 isbn-13 : 978-1492913733 for my family , who encouraged me to never stop fighting for my dreams chapter 1 summer vacations supposed to be fun , right ?',
  'i wish i had a better answer to that question .',
  'starlings , new york is not the place youd expect much to happen .',
  'its a small quiet town , the kind where everyone knows your name .',
  'its a place where your parents wouldnt even care if you stayed out late biking with your friends .',
  'only because everyone felt so safe , so comfy .',
  'they dont know the half of it .',
  'but i do .',
  'i know it all and starlings is not the place where you want to be after dark .']}

In [8]:
import numpy as np
import random

simple_stories_examples = []
simple_wiki_examples = []
bookcorpus_examples = []

for sample in simple_stories_ds["train"]:
    simple_stories_examples.append(sample["story"])
    if len(simple_stories_examples) == 45000:
        break
print("Collected Simple Stories Examples")

for sample in simple_wiki_ds["train"]:
    simple_wiki_examples.append("# Title: "+sample["title"]+"\n"+sample["text"])
    if len(simple_wiki_examples) == 30000:
        break
print("Collected Simple Wiki Examples")

for i in range(0, len(bookcorpus_ds["train"]), 10):

    sample = [x for x in bookcorpus_ds["train"][i:i+10]["text"]]

    bookcorpus_examples.append("\n".join(sample))

    if len(bookcorpus_examples) == 25000:
        break
print("Collected BookCoupus Examples")


print("Creating interleaved data")

choices = (["a"]*len(simple_stories_examples)) + (["b"]*len(simple_wiki_examples)) + (["c"]*len(bookcorpus_examples))

random.shuffle(choices)

data = []

for choice in choices:
    if choice == "a":
        data.append(simple_stories_examples.pop(0))
    elif choice == "b":
        data.append(simple_wiki_examples.pop(0))
    else:
        data.append(bookcorpus_examples.pop(0))

print("Interleaved dataset created")





Collected Simple Stories Examples
Collected Simple Wiki Examples
Collected BookCoupus Examples
Creating interleaved data
Interleaved dataset created


In [10]:
from tqdm.auto import tqdm
idx = 0
eot = tokenizer.eot_token
# pbar = tqdm(total = len(data), )
total_tokens = 0
for text in tqdm(data, desc="Calculating token counts"):
    # Tokenize + 1 for the <|endoftext|> delimiter
    tokens = tokenizer.encode(text, allowed_special="all")
    total_tokens += len(tokens) + 1

print(f"Total tokens: {total_tokens:,}")

mmap = np.memmap("mid_training.npy", dtype=np.uint16, mode="w+", shape=(total_tokens, ))
for i in tqdm(range(1, len(data)), desc="Creating tokens"):
    text = data[i]
    tokens = tokenizer.encode(text, allowed_special="all")
    tokens.append(eot)

    length = len(tokens)
    mmap[idx: idx+length] = np.array(tokens, dtype=np.uint16)
    idx += length

mmap.flush()
print("Successfullly saved tokens.")

Calculating token counts:   0%|          | 0/100000 [00:00<?, ?it/s]

Total tokens: 24,512,087


Creating tokens:   0%|          | 0/99999 [00:00<?, ?it/s]

Successfullly saved tokens.


### Creating dataloaders

In [7]:
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
class GSMDataset(Dataset):
    def __init__(self, data, block_size):
        self.data = data
        self.block_size = block_size

    def __len__(self):
        return len(self.data) - self.block_size

    def __getitem__(self, idx):
        x = self.data[idx:idx+self.block_size]
        y = self.data[idx+1:idx+self.block_size+1]

        return x, y


def create_data_doc(data):
    doc = ""
    pbar = tqdm(total=len(data))
    for sample in data:
        text = sample["story"]
        doc += f"{text}\n{tokenizer.decode([tokenizer.eot_token])}\n\n"
        pbar.update(1)


    return doc.strip()

In [8]:
ds["train"][0]["story"]+tokenizer.decode([tokenizer.eot_token])

"Eagerly, a girl named Kim went hiking with her friends. They found a secret passageway under a big rock. Curious, they crawled through it and entered a magical world where all the animals talked. They learned that the animals had their own struggles but always found a way to be happy. \n\nInspired, the children played with the animals and shared their own stories of hardship. They realized that laughter could help them through hard times. When they finally left, they took the animals' smiles with them, knowing that happiness can come from the most unexpected places.<|endoftext|>"

In [ ]:
train_data = torch.tensor(tokenizer.encode(create_data_doc(ds["train"]), allowed_special="all"), dtype=torch.long, device=device)
test_data = torch.tensor(tokenizer.encode(create_data_doc(ds["test"]), allowed_special="all"), dtype=torch.long, device=device)

train_dataset = GSMDataset(train_data, config.block_size)
test_dataset = GSMDataset(test_data, config.block_size)

train_dataloader = DataLoader(train_dataset, batch_size=config.batch_size, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size=config.batch_size, shuffle=True)

  0%|          | 0/2115696 [00:00<?, ?it/s]

In [ ]:
print(len(test_dataset))

## Training loop

In [ ]:
model = CustomGPT(config)
state_dict = torch.load("tara_n1_pretrain_v2.pth", map_location=device)
state_dict = {k.removeprefix("module."):v for k, v in state_dict.items()}
model.load_state_dict(state_dict)
model.to(device)

loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
scaler = torch.amp.GradScaler()

In [ ]:
from tqdm.auto import tqdm

steps = 2065
test_count = 500
test_steps = 20

train_iter = iter(train_dataloader)

avg_train_loss = 0

for step in tqdm(range(1, steps+1)):
    model.train()
    if not train_iter:
        train_iter = iter(train_dataloader)

    train_loss = train_step(model, train_iter, loss_fn, optimizer, scaler, device, accumulation_steps=1)
    avg_train_loss += train_loss
    if step%test_count == 0:
        avg_train_loss /= test_count

        test_loss = test_step(model, test_dataloader, test_steps, loss_fn, device)

        print(f"Step: {step} | Train loss: {avg_train_loss} | Test loss: {test_loss}")
        avg_train_loss = 0


In [ ]:
model.eval()

query = "Once upon a time, there live a king "
context = torch.tensor(tokenizer.encode(query), dtype=torch.long, device=device).unsqueeze(0)

with torch.inference_mode():
    generated = model.generate(context)

print(tokenizer.decode(generated[0].tolist()).split("<|endoftext|>")[0].replace("&amp;", "&").replace("&lt;", "<").replace("&gt;", ">"))
